# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

Runs **FastAPI backend + Qwen 3.5 4B** in Google Colab.
The model is downloaded during the installation phase.

In [ ]:
# CELL 1: Clean and uninstall existing (to avoid version conflicts with Colab default)
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Environment cleaned")

In [ ]:
# CELL 2: Install Libraries + Qwen 3.5 4B Model weights
# This step acts as your 'installation' of everything including the model
!pip install fastapi==0.124.1 uvicorn==0.34.0 starlette==0.49.1 pydantic==2.12.0 pydantic-settings httpx==0.28.1 python-multipart==0.0.18 neo4j==5.23.0 pyngrok PyMuPDF==1.24.0 pdfplumber==0.11.0 transformers==4.45.0 accelerate==0.34.0 torch

print("\n📦 Downloading Qwen 3.5 4B Model (this may take 2-4 minutes)...")
# Use a tiny python script to pre-download the weights to cache
!python -c "from transformers import AutoModelForCausalLM, AutoTokenizer; mid='Qwen/Qwen3.5-4B'; AutoTokenizer.from_pretrained(mid, trust_remote_code=True); AutoModelForCausalLM.from_pretrained(mid, trust_remote_code=True)"

print("\n✅ ALL INSTALLED: Libraries + Qwen 3.5 4B weights ready")

In [ ]:
# CELL 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/Knowledge_graph'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Working in {PROJECT_ROOT}")

In [ ]:
# CELL 4: Quick load model into GPU memory
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"✅ Model loaded on: {model.device}")

In [ ]:
# CELL 5: Start uvicorn with nohup
# (Setting PYTHONPATH ensures uvicorn finds your code)
os.environ['NEO4J_URI'] = 'neo4j+s://xxxxxxxx.databases.neo4j.io'
os.environ['NEO4J_USER'] = 'neo4j'
os.environ['NEO4J_PASSWORD'] = 'your-password'

!pkill -f uvicorn 2>/dev/null
!nohup bash -c "cd {PROJECT_ROOT} && PYTHONPATH={PROJECT_ROOT}:$PYTHONPATH uvicorn backend.main:app --host 0.0.0.0 --port 8000" > {PROJECT_ROOT}/backend.log 2>&1 &
print("🚀 Backend started in background. Check backend.log")